Computing VPD files from the bias corrected inputs using the functions defined in VPD_functions.py

In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client(n_workers=7, threads_per_worker=1) 
#client = Client()

client

/g/data/xp65/public/apps/med_conda/envs/analysis3-24.12/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42021 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/42021/status,
Dashboard: /proxy/42021/status,Workers: 7
Total threads: 7,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43411,Workers: 7
Dashboard: /proxy/42021/status,Total threads: 7
Started: Just now,Total memory: 251.19 GiB
Comm: tcp://127.0.0.1:37893,Total threads: 1
Dashboard: /proxy/41579/status,Memory: 35.88 GiB
Nanny: tcp://127.0.0.1:42513,


2025-08-27 13:08:32,285 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/ACCESS-ESM1-5/ssp370/r6i1p1f1/BARPA-R/v1-r1/day/ssp370_ACCESS-ESM1-5_BARPA-R_MRNBC_gwl2.0_vpd.nc', lease_id='0d07c70c85f549d788713404f0bcb596'. This can happen if the Lock or Semaphore timed out before.
2025-08-27 13:09:48,490 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/ACCESS-ESM1-5/ssp370/r6i1p1f1/BARPA-R/v1-r1/day/ssp370_ACCESS-ESM1-5_BARPA-R_MRNBC_gwl2.0_vpd.nc', lease_id='7ab1839ef28844729ff35a4d143b954a'. This can happen if the Lock or Semaphore timed out before.
2025-08-27 13:11:14,602 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/ACCESS-ESM1-5/ssp370/r6i1p1f1/BARPA-R/v1-r1/day/ssp370_ACCESS-ESM1-5_BARPA-R_MRNBC_gwl2.0_vpd.nc', lease_id='81

In [2]:
#import all the stuff
from netCDF4 import Dataset
import xarray as xr
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import glob
import sys
sys.path.append("/g/data/mn51/users/nb6195/project/vpd/VPD")
import VPD_functions as VPD
sys.path.append("/g/data/mn51/users/nb6195/project/gwls/")
import gwl

In [19]:
#Set parameters
CMIP='CMIP6'
AGENCY = 'CSIRO' 
RCM = 'CCAM-v2203-SN'
#AGENCY = 'BOM' 
#RCM = 'BARPA-R'

#GCM = 'ACCESS-CM2' ensemble = 'r4i1p1f1' #Done
GCM = 'ACCESS-ESM1-5' 
ensemble = 'r6i1p1f1' #Done
#GCM = 'EC-Earth3' ensemble = 'r1i1p1f1' #Done
#GCM = 'MPI-ESM1-2-HR' ensemble = 'r1i1p1f1' #BOM done, no CSIRO
#GCM = 'CESM2' ensemble = 'r11i1p1f1' #Done
#GCM = 'CMCC-ESM2' ensemble = 'r1i1p1f1' #Done
#GCM = 'NorESM2-MM' ensemble = 'r1i1p1f1' #Done
#GCM = 'CNRM-ESM2-1' ensemble = 'r1i1p1f2' #CSIRO Done, no BOM

#pathway = 'ssp126'
pathway = 'ssp370'

bc_method = 'MRNBC'
#bc_method = 'QME'

ddir = f"/g/data/kj66/CORDEX/output/{CMIP}/bias-adjusted-output/AUST-05i/{AGENCY}/{GCM}"
output_dir = '/g/data/ia39/ncra/bushfire/vpd/'
output_dir_mm = '/g/data/ia39/ncra/bushfire/vpd/monthly_mean/'

In [20]:
var1 = 'tasmaxAdjust'
var2 = 'hursminAdjust'

In [21]:
#read in files

#tasmax
infiles1a=glob.glob(ddir+f'/historical/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var1}/v20241216/{var1}_AUST-05i_{GCM}_historical_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
infiles1b=glob.glob(ddir+f'/{pathway}/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var1}/v20241216/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
tasmax_master_ds = xr.open_mfdataset(infiles1a + infiles1b)

#hursmin
infiles2a=glob.glob(ddir+f'/historical/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var2}/v20241216/{var2}_AUST-05i_{GCM}_historical_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
infiles2b=glob.glob(ddir+f'/{pathway}/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var2}/v20241216/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
hursmin_master_ds = xr.open_mfdataset(infiles2a + infiles2b)

In [22]:
#Extract time period corresponding to the chosen GWL for tasmax and rh
chosen_gwl = '1.2'

gwl_tasmax = gwl.get_GWL_timeslice(tasmax_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var1]
gwl_rh = gwl.get_GWL_timeslice(hursmin_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var2]

In [25]:
#Create the datasets for vpd and monthly mean vpd
gwl_vpd, monthly_mean_vpd = VPD.vpd_calc(gwl_rh, gwl_tasmax, chosen_gwl)

In [27]:
#print vpd to an external file
file_name_vpd = pathway + '_' + GCM + '_' + RCM + '_' + bc_method + '_gwl' + chosen_gwl + '_vpd.nc'
output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
gwl_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

HDF5-DIAG: Error detected in HDF5 (1.14.3) thread 0:
  #000: H5F.c line 836 in H5Fopen(): unable to synchronously open file
    major: File accessibility
    minor: Unable to open file
  #001: H5F.c line 796 in H5F__open_api_common(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #002: H5VLcallback.c line 3863 in H5VL_file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #003: H5VLcallback.c line 3675 in H5VL__file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #004: H5VLnative_file.c line 128 in H5VL__native_file_open(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #005: H5Fint.c line 1910 in H5F_open(): unable to lock the file
    major: File accessibility
    minor: Unable to lock file
  #006: H5FD.c line 2412 in H5FD_lock(): driver lock request failed
    major: Virtual File Layer
    minor: Unable to lock file
  #007: H5FDsec2.c line 941 

OSError: [Errno -101] NetCDF: HDF error: '/g/data/ia39/ncra/bushfire/vpd/ACCESS-ESM1-5/ssp370/r6i1p1f1/CCAM-v2203-SN/v1-r1/day/ssp370_ACCESS-ESM1-5_CCAM-v2203-SN_MRNBC_gwl1.2_vpd.nc'

In [ ]:
#print monthly mean ds to external file

file_name_mean = 'gwl' + chosen_gwl + '_monthly_mean_vpd_' + GCM + '_' + RCM + '_' + bc_method + '_' + pathway + '_' + ensemble + '.nc' 
output_file_location = output_dir_mm + file_name_mean
monthly_mean_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

HDF5-DIAG: Error detected in HDF5 (1.14.3) thread 0:
  #000: H5F.c line 836 in H5Fopen(): unable to synchronously open file
    major: File accessibility
    minor: Unable to open file
  #001: H5F.c line 796 in H5F__open_api_common(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #002: H5VLcallback.c line 3863 in H5VL_file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #003: H5VLcallback.c line 3675 in H5VL__file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #004: H5VLnative_file.c line 128 in H5VL__native_file_open(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #005: H5Fint.c line 1910 in H5F_open(): unable to lock the file
    major: File accessibility
    minor: Unable to lock file
  #006: H5FD.c line 2412 in H5FD_lock(): driver lock request failed
    major: Virtual File Layer
    minor: Unable to lock file
  #007: H5FDsec2.c line 941 